# 🏮 Pasar Malam Vendor Revenue Prediction

This notebook walks through a **machine learning regression pipeline** that predicts daily revenue for vendors at digitised Pasar Malam (night market) events in Singapore.

### Pipeline Overview
1. **Generate** realistic synthetic training data
2. **Train** a Gradient Boosting regression model
3. **Predict** tomorrow's revenue for different vendor scenarios

### Features Used (8 total)
| # | Feature | Description |
|---|---------|-------------|
| 1 | `week_day` | Day of the week (1=Mon … 7=Sun) |
| 2 | `is_rain` | Whether it rains (0/1) |
| 3 | `is_public_holiday` | Whether it's a public holiday (0/1) |
| 4 | `is_food_vendor` | Food vendor (1) or non-food vendor (0) |
| 5 | `temperature` | Temperature in °C |
| 6 | `yesterday_customer_count` | Number of customers the previous day |
| 7 | `yesterday_revenue` | Revenue from the previous day ($) |
| 8 | `event_location_rating` | Location quality rating (1–10) |

---
## Step 1: Generate Synthetic Training Data

Since we don't have real historical data yet, we generate **5,000 realistic samples** using domain knowledge about Pasar Malam economics:

- **Non-food vendors** average **$4,000/day**
- **Food vendors** earn **50% more** (~$6,000/day) because they attract more customers
- Food customers spend ~**$7 each**, non-food customers spend ~**$20 each**
- **Weekends & Fridays** have significantly higher foot traffic
- **Rain** reduces revenue by 25%
- **Public holidays** boost revenue by 30%
- **Temperature** has an optimal point (29°C); deviations reduce revenue
- **Location rating** scales revenue ±5% per point from baseline (5)
- **10% random noise** is added to simulate real-world variance

In [ ]:
import numpy as np
import pandas as pd
import random

# ============================================================
# CONFIGURABLE PARAMETERS
# ============================================================

NUM_SAMPLES = 5000
RANDOM_SEED = 42

# --- Revenue Baselines ---
NON_FOOD_AVG_DAILY_REVENUE = 4000
FOOD_REVENUE_MULTIPLIER = 1.5

# --- Spend Per Customer ---
FOOD_SPEND_PER_CUSTOMER = 7
NON_FOOD_SPEND_PER_CUSTOMER = 20

# --- Day-of-Week Traffic Multipliers (Mon=1 to Sun=7) ---
DAY_MULTIPLIERS = {
    1: 0.70,   # Monday
    2: 0.75,   # Tuesday
    3: 0.80,   # Wednesday
    4: 0.85,   # Thursday
    5: 1.20,   # Friday
    6: 1.40,   # Saturday
    7: 1.30,   # Sunday
}

# --- Weather & Environment ---
RAIN_REVENUE_PENALTY = 0.75
HOLIDAY_REVENUE_BOOST = 1.30
TEMP_MIN = 25
TEMP_MAX = 35
TEMP_OPTIMAL = 29
TEMP_PENALTY_PER_DEGREE = 0.02
LOCATION_RATING_MIN = 1
LOCATION_RATING_MAX = 10
LOCATION_RATING_BASELINE = 5
LOCATION_BOOST_PER_POINT = 0.05
REVENUE_NOISE_STD = 0.10

### Data Generation Logic

For each sample, we:
1. Randomly pick values for all 8 features
2. Calculate base revenue depending on vendor type
3. Apply multipliers for day-of-week, rain, holiday, temperature, and location
4. Add Gaussian noise to simulate real-world randomness
5. Derive yesterday's revenue and customer count from today's values (with some variation)

In [ ]:
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

records = []

for _ in range(NUM_SAMPLES):
    # Generate feature values
    week_day = random.randint(1, 7)
    is_rain = random.choices([0, 1], weights=[0.65, 0.35])[0]
    is_public_holiday = random.choices([0, 1], weights=[0.92, 0.08])[0]
    is_food_vendor = random.choices([0, 1], weights=[0.40, 0.60])[0]
    temperature = round(np.random.uniform(TEMP_MIN, TEMP_MAX), 1)
    event_location_rating = random.randint(LOCATION_RATING_MIN, LOCATION_RATING_MAX)

    # Calculate base revenue
    if is_food_vendor:
        base_revenue = NON_FOOD_AVG_DAILY_REVENUE * FOOD_REVENUE_MULTIPLIER
    else:
        base_revenue = NON_FOOD_AVG_DAILY_REVENUE

    # Apply all multipliers
    revenue = base_revenue * DAY_MULTIPLIERS[week_day]
    if is_rain:
        revenue *= RAIN_REVENUE_PENALTY
    if is_public_holiday:
        revenue *= HOLIDAY_REVENUE_BOOST

    temp_deviation = abs(temperature - TEMP_OPTIMAL)
    revenue *= 1.0 - (temp_deviation * TEMP_PENALTY_PER_DEGREE)

    rating_diff = event_location_rating - LOCATION_RATING_BASELINE
    revenue *= 1.0 + (rating_diff * LOCATION_BOOST_PER_POINT)

    # Add noise
    revenue *= np.random.normal(1.0, REVENUE_NOISE_STD)
    revenue = max(round(revenue, 2), 0)

    # Derive yesterday's values
    yesterday_revenue = round(revenue * np.random.uniform(0.80, 1.20), 2)
    if is_food_vendor:
        yesterday_customers = round(yesterday_revenue / FOOD_SPEND_PER_CUSTOMER)
    else:
        yesterday_customers = round(yesterday_revenue / NON_FOOD_SPEND_PER_CUSTOMER)

    records.append({
        "week_day": week_day,
        "is_rain": is_rain,
        "is_public_holiday": is_public_holiday,
        "is_food_vendor": is_food_vendor,
        "temperature": temperature,
        "yesterday_customer_count": yesterday_customers,
        "yesterday_revenue": yesterday_revenue,
        "event_location_rating": event_location_rating,
        "today_revenue": revenue,
    })

df = pd.DataFrame(records)
print(f"Generated {NUM_SAMPLES} samples")
print(f"\nMean revenue (all):      ${df['today_revenue'].mean():.2f}")
print(f"Mean revenue (food):     ${df[df['is_food_vendor']==1]['today_revenue'].mean():.2f}")
print(f"Mean revenue (non-food): ${df[df['is_food_vendor']==0]['today_revenue'].mean():.2f}")

### Preview the Generated Data

In [ ]:
df.head(10)

### Data Distribution Visualisation

Let's look at how revenue distributes across vendor types and days of the week.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Revenue distribution
axes[0].hist(df['today_revenue'], bins=50, color='#FF6B6B', edgecolor='white')
axes[0].set_title('Revenue Distribution')
axes[0].set_xlabel('Revenue ($)')
axes[0].set_ylabel('Count')

# Revenue by vendor type
df.boxplot(column='today_revenue', by='is_food_vendor', ax=axes[1])
axes[1].set_title('Revenue by Vendor Type')
axes[1].set_xlabel('Is Food Vendor (0=No, 1=Yes)')
axes[1].set_ylabel('Revenue ($)')
plt.suptitle('')

# Revenue by day of week
day_labels = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
day_means = [df[df['week_day']==d]['today_revenue'].mean() for d in range(1,8)]
axes[2].bar(day_labels, day_means, color='#4ECDC4', edgecolor='white')
axes[2].set_title('Average Revenue by Day of Week')
axes[2].set_ylabel('Avg Revenue ($)')

plt.tight_layout()
plt.show()

---
## Step 2: Train the Regression Model

### Why Gradient Boosting over Linear Regression?

Our data has **non-linear relationships** and **feature interactions**:
- Rain on a Saturday hurts more than rain on a Monday (interaction effect)
- Temperature has a sweet spot at 29°C — not a straight line
- Location rating has diminishing returns at the top end

**Gradient Boosting** builds sequential decision trees that naturally capture these patterns, while Linear Regression can only model straight-line additive effects.

We use an **80/20 train-test split** to evaluate how well the model generalises to unseen data.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ============================================================
# CONFIGURABLE PARAMETERS
# ============================================================

TEST_SIZE = 0.20
RANDOM_SEED = 42
MODEL_TYPE = "gradient_boosting"  # change to "linear" to compare

# Gradient Boosting hyperparameters
GB_N_ESTIMATORS = 200
GB_MAX_DEPTH = 4
GB_LEARNING_RATE = 0.1

FEATURE_COLUMNS = [
    "week_day",
    "is_rain",
    "is_public_holiday",
    "is_food_vendor",
    "temperature",
    "yesterday_customer_count",
    "yesterday_revenue",
    "event_location_rating",
]
TARGET_COLUMN = "today_revenue"

X = df[FEATURE_COLUMNS]
y = df[TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED
)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples:  {len(X_test)}")

### Train the Model

We fit a **Gradient Boosting Regressor** with 200 trees, max depth 4, and a learning rate of 0.1.

In [ ]:
if MODEL_TYPE == "gradient_boosting":
    model = GradientBoostingRegressor(
        n_estimators=GB_N_ESTIMATORS,
        max_depth=GB_MAX_DEPTH,
        learning_rate=GB_LEARNING_RATE,
        random_state=RANDOM_SEED,
    )
    print("Training: Gradient Boosting Regressor")
else:
    model = LinearRegression()
    print("Training: Linear Regression")

model.fit(X_train, y_train)
print("Model trained successfully!")

### Model Evaluation

We evaluate using three metrics:
- **MAE** (Mean Absolute Error): Average dollar amount the prediction is off by
- **RMSE** (Root Mean Squared Error): Penalises large errors more heavily
- **R²** (Coefficient of Determination): How much variance the model explains (1.0 = perfect)

In [ ]:
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE  (Mean Absolute Error): ${mae:.2f}")
print(f"RMSE (Root Mean Sq Error):  ${rmse:.2f}")
print(f"R²   (Coefficient of Det.): {r2:.4f}")

### Feature Importances

This shows which features the model relies on most when making predictions. Higher = more influential.

In [ ]:
if hasattr(model, "feature_importances_"):
    importances = sorted(zip(FEATURE_COLUMNS, model.feature_importances_), key=lambda x: -x[1])
    names, values = zip(*importances)

    plt.figure(figsize=(10, 5))
    plt.barh(names[::-1], values[::-1], color='#FF6B6B', edgecolor='white')
    plt.xlabel('Importance')
    plt.title('Feature Importances — Gradient Boosting Model')
    plt.tight_layout()
    plt.show()

    for name, imp in importances:
        print(f"  {name:30s} {imp:.4f}")

### Actual vs Predicted — Scatter Plot

A good model should cluster points tightly along the diagonal line (predicted ≈ actual).

In [ ]:
plt.figure(figsize=(7, 7))
plt.scatter(y_test, y_pred, alpha=0.3, s=10, color='#4ECDC4')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect prediction')
plt.xlabel('Actual Revenue ($)')
plt.ylabel('Predicted Revenue ($)')
plt.title('Actual vs Predicted Revenue')
plt.legend()
plt.tight_layout()
plt.show()

---
## Step 3: Predict Tomorrow's Revenue

Now we use the trained model to predict revenue for different vendor scenarios. Edit the values below to try your own!

In [ ]:
# ============================================================
# EDIT THESE SCENARIOS TO PLAY WITH PREDICTIONS
# ============================================================

scenarios = [
    {
        "label": "Saturday, Food Vendor, Good Location, No Rain",
        "week_day": 6,                    # 1=Mon ... 7=Sun
        "is_rain": 0,                     # 0=No, 1=Yes
        "is_public_holiday": 0,           # 0=No, 1=Yes
        "is_food_vendor": 1,              # 0=Non-food, 1=Food
        "temperature": 29.0,              # Celsius
        "yesterday_customer_count": 800,
        "yesterday_revenue": 5600.00,
        "event_location_rating": 8,       # 1 to 10
    },
    {
        "label": "Monday, Non-Food Vendor, Rain, Low Rating",
        "week_day": 1,
        "is_rain": 1,
        "is_public_holiday": 0,
        "is_food_vendor": 0,
        "temperature": 32.0,
        "yesterday_customer_count": 120,
        "yesterday_revenue": 2400.00,
        "event_location_rating": 3,
    },
    {
        "label": "Friday, Food Vendor, Public Holiday, Great Location",
        "week_day": 5,
        "is_rain": 0,
        "is_public_holiday": 1,
        "is_food_vendor": 1,
        "temperature": 28.5,
        "yesterday_customer_count": 1000,
        "yesterday_revenue": 7000.00,
        "event_location_rating": 9,
    },
    {
        "label": "Wednesday, Non-Food Vendor, No Rain, Average Location",
        "week_day": 3,
        "is_rain": 0,
        "is_public_holiday": 0,
        "is_food_vendor": 0,
        "temperature": 30.0,
        "yesterday_customer_count": 180,
        "yesterday_revenue": 3600.00,
        "event_location_rating": 5,
    },
]

print("=" * 60)
print("  PASAR MALAM VENDOR REVENUE PREDICTIONS")
print("=" * 60)

for scenario in scenarios:
    label = scenario["label"]
    features = {k: scenario[k] for k in FEATURE_COLUMNS}
    input_df = pd.DataFrame([features])
    predicted_revenue = model.predict(input_df)[0]

    print(f"\n📍 {label}")
    print(f"   💰 Predicted Revenue: ${predicted_revenue:,.2f}")

print("\n" + "=" * 60)

---
## Summary

| Step | What it does |
|------|-------------|
| **Data Generation** | Creates 5,000 realistic synthetic samples based on Pasar Malam economics |
| **Model Training** | Trains a Gradient Boosting Regressor on 80% of the data |
| **Evaluation** | Tests on the remaining 20% using MAE, RMSE, and R² |
| **Prediction** | Uses the model to forecast tomorrow's revenue for any vendor scenario |

### Key Insights
- **Yesterday's revenue** and **customer count** are the strongest predictors (recent momentum matters)
- **Vendor type** (food vs non-food) significantly affects revenue levels
- **Day of week** captures the Friday/weekend traffic surge
- **Rain** is the biggest negative factor — consider tent/shelter investments

### Next Steps
- Replace synthetic data with **real vendor data** from the app once available
- Add more features (e.g. nearby competing events, social media buzz)
- Retrain periodically as more data accumulates